In [ ]:
#Turns out, you have to use python 3.10 in order for this all to work. The older femr model must also be used
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-venv
!python3.10 -m venv /content/femr_venv
!/content/femr_venv/bin/pip install "femr[models]==0.1.16"

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
python3.10 is already the newest version (3.10.12-1~22.04.15).
python3.10-venv is already the 

In [ ]:
#Libraries on which femr work depends
!/content/femr_venv/bin/pip install "jax[cpu]==0.4.8" "jaxlib==0.4.7" \
  -f https://storage.googleapis.com/jax-releases/jax_releases.html
!/content/femr_venv/bin/pip install "scipy<1.11"

Looking in links: https://storage.googleapis.com/jax-releases/jax_releases.html
DEPRECATION: The HTML index page being used (https://storage.googleapis.com/jax-releases/jax_releases.html) is not a proper HTML 5 document. This is in violation of PEP 503 which requires these pages to be well-formed HTML 5 documents. Please reach out to the owners of this index page, and ask them to update this index page to a valid HTML 5 document. pip 22.2 will enforce this behaviour change. Discussion can be found at https://github.com/pypa/pip/issues/10825
  Using cached jax-0.4.8-py3-none-any.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.1/66.1 MB 6.9 MB/s eta 0:00:00
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.4.23
    Uninstalling jaxlib-0.4.23:
      Successfully uninstalled jaxlib-0.4.23
  Attempting uninstall: jax
    Found existing installation: jax 0.4.23
    Uninstalling jax-0.4.23:
      Successfully uninstalled jax-0.4.23


In [ ]:
!/content/femr_venv/bin/python -c "import jax.numpy as jnp; jnp.DeviceArray = jnp.ndarray; import femr.models.transformer; print('SUCCESS')"

SUCCESS


In [ ]:
!gdown https://drive.google.com/uc?id=1Yhsz47BG8SZGoIFx0Q_UGEKsfhi2aiDs -O synthetic_data.zip
!unzip -oq synthetic_data.zip

Downloading...
From: https://drive.google.com/uc?id=1Yhsz47BG8SZGoIFx0Q_UGEKsfhi2aiDs
To: /content/synthetic_data.zip
100% 22.6M/22.6M [00:00<00:00, 162MB/s]


In [ ]:
!ls synthetic_data

clmbr  extract_lite  motor  train_person_ids.csv


In [ ]:
%%bash
python inspect_motor_dictionary.py //root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc

python3: can't open file '/content/inspect_motor_dictionary.py': [Errno 2] No such file or directory


CalledProcessError: Command 'b'python inspect_motor_dictionary.py //root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc\n'' returned non-zero exit status 2.

In [ ]:
#Here you paste your token for access to the MOTOR t-base model: hf hmmylyWlwXUbNjZRLrxiqdtPhHuUJRHfmP
from huggingface_hub import login
login()

In [ ]:
#This downloads the model into this colab environment; OMG it worked.
from huggingface_hub import snapshot_download
motor_path = snapshot_download("StanfordShahLab/motor-t-base")
print(motor_path)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc


In [ ]:
#This is making inspect_motor_dictionary.py. You don't have to run this. You have the text file this makes, and can search
#that
%%writefile inspect_motor_dictionary.py
"""
Dump the code strings that the real motor-t-base model actually recognizes.

run this FIRST, copy a handful of real codes out of the printout, and use
those when you author your simulated patients in make_patients.py.

USAGE (in your Colab, with motor-t-base cloned locally):
    python inspect_motor_dictionary.py /path/to/motor-t-base

If you cloned the HF repo, MODEL_PATH is the repo dir (it contains `dictionary`
and `model`).
"""

import os
import sys
import json
import collections

import msgpack  # femr depends on msgpack>=1.0.5, so it's already installed


def load_dictionary(model_path: str):
    dict_path = os.path.join(model_path, "dictionary")
    if not os.path.exists(dict_path):
        # some checkpoints nest it; try a couple of fallbacks
        for cand in ("dictionary", "clmbr_dictionary", "survival_dictionary"):
            p = os.path.join(model_path, cand)
            if os.path.exists(p):
                dict_path = p
                break
    if not os.path.exists(dict_path):
        raise FileNotFoundError(
            f"No `dictionary` file found under {model_path!r}. "
            f"Contents: {os.listdir(model_path)}"
        )
    with open(dict_path, "rb") as f:
        return msgpack.load(f, raw=False)


def find_code_strings(obj):
    """The femr dictionary is a nested msgpack structure. We don't hard-code its
    exact schema (it has varied across versions); instead we walk it and collect
    every value that looks like a 'Vocab/code' string, plus anything stored under
    a key literally named 'code_string'."""
    codes = collections.Counter()

    def walk(o):
        if isinstance(o, dict):
            for k, v in o.items():
                if k == "code_string" and isinstance(v, str):
                    codes[v] += 1
                walk(v)
        elif isinstance(o, (list, tuple)):
            for v in o:
                walk(v)
        elif isinstance(o, str):
            # heuristic: real femr codes look like "SNOMED/12345", "RxNorm/..."
            if "/" in o and " " not in o and len(o) < 64:
                codes[o] += 1

    walk(obj)
    return codes


def main():
    if len(sys.argv) < 2:
        print(__doc__)
        sys.exit(1)
    model_path = sys.argv[1]

    d = load_dictionary(model_path)

    print("=" * 70)
    print("TOP-LEVEL STRUCTURE OF THE DICTIONARY")
    print("=" * 70)
    if isinstance(d, dict):
        for k, v in d.items():
            kind = type(v).__name__
            n = len(v) if hasattr(v, "__len__") else "-"
            print(f"  key={k!r:30s} type={kind:8s} len={n}")
    else:
        print(f"  (top level is a {type(d).__name__})")

    codes = find_code_strings(d)
    print()
    print("=" * 70)
    print(f"FOUND {len(codes)} DISTINCT 'Vocab/code'-style strings")
    print("=" * 70)

    # group by vocabulary prefix so you can see what families are available
    by_vocab = collections.Counter(c.split("/", 1)[0] for c in codes)
    print("Vocabularies present (prefix -> count):")
    for vocab, n in by_vocab.most_common():
        print(f"  {vocab:20s} {n}")

    print()
    print("Sample codes you can safely use (first 40):")
    for c, _ in list(codes.most_common())[:40]:
        print("   ", c)

    # also write the full list to disk so you can grep it
    out = os.path.join(os.getcwd(), "motor_dictionary_codes.txt")
    with open(out, "w") as f:
        for c in sorted(codes):
            f.write(c + "\n")
    print()
    print(f"Full code list written to: {out}")
    print("Pick demographics (Birth/…, Gender/…, Race/…) + a few clinical codes "
          "from this list for make_patients.py.")


if __name__ == "__main__":
    main()

Overwriting inspect_motor_dictionary.py


In [ ]:
#This will show you the different prefixes in the motor_dictionary_codes document you now have
!python inspect_motor_dictionary.py /root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc

TOP-LEVEL STRUCTURE OF THE DICTIONARY
  key='age_stats'                    type=dict     len=2
  key='all_parents'                  type=dict     len=162227
  key='ontology_rollup'              type=list     len=1812760
  key='regular'                      type=list     len=0

FOUND 162049 DISTINCT 'Vocab/code'-style strings
Vocabularies present (prefix -> count):
  ICD10PCS             59288
  SNOMED               37106
  LOINC                26888
  RxNorm               21824
  CPT4                 11627
  ICD9Proc             1866
  ICDO3                1092
  RxNorm Extension     1046
  CARE_SITE            429
  HemOnc               266
  Cancer Modifier      220
  HCPCS                217
  CVX                  64
  SNOMED Veterinary    42
  OMOP Extension       25
  PPI                  18
  Visit                6
  ATC                  5
  Race                 5
  CMS Place of Service 4
  Ethnicity            2
  Gender               2
  Domain               1
  Condition Type 

In [ ]:
#This is making a file simple_femr_input/simulated_patients.csv.
%%writefile make_patients.py
"""
Here I am generating 1-2 SIMULATED patients in the "simple femr" CSV format, which the
`etl_simple_femr` tool turns into a femr PatientDatabase.

This mirrors the official FEMR generator
(tutorials/synthetic_data_generation/generate_simple_femr.py) and the schema
enforced by femr.etl_pipelines.simple, so the output is byte-compatible with the
real ETL.

KEY RULES (enforced by femr.etl_pipelines.simple.convert_file_to_event_file):
  * Header MUST be: patient_id,start,code,value,dosage,visit_ids,lab_units,clarity_source
      - patient_id, start, code, value are special.
      - Every OTHER column (dosage, visit_ids, lab_units, clarity_source) is
        attached to the Event as metadata. Empty string -> None.
  * `code` MUST contain a "/" (Vocabulary/code), e.g. "ICD10CM/E11.4".
    The ETL asserts this.
  * `start` must be ISO format (YYYY-MM-DD or full ISO datetime).
  * Events do NOT need to be pre-sorted; the ETL sorts by (patient_id, start).
"""
import csv
import datetime
import os

# --------------------------------------------------------------------------- #
# CONFIG  --  edit these                                                       #
# --------------------------------------------------------------------------- #
OUT_DIR = "simple_femr_input"          # folder of CSV(s) you'll feed to etl_simple_femr
OUT_FILE = "simulated_patients.csv"

# >>> REPLACE every code below with one you confirmed is in the model dictionary
#     via inspect_motor_dictionary.py. These are PLACEHOLDERS in the same SHAPE
#     as real femr/OMOP codes; they will be dropped by the real model.        <
DEMOGRAPHIC_CODES = {
    "birth":  "Birth/Birth",          # nearly always present
}
# A small clinical vocabulary to draw from. Delete this eventually as you transition to making more patients like patient 1.
DIAGNOSIS_CODES = ["ICD10CM/E11.4", "ICD10CM/I10"]      # type 2 diabetes, HTN (placeholders)
PROCEDURE_CODES = ["CPT/99213"]                          # office visit (placeholder)
DRUG_CODES      = ["Drug/Atorvastatin", "Drug/Metformin"]

#Just the way you wrote this, you can reference this dict when making patient events below like so: VITALS_CODES["systolic_bp"]
VITALS_CODES = {
    "systolic_bp":      "LOINC/8480-6",    # mmHg
    "diastolic_bp":     "LOINC/8462-4",    # mmHg
    "heart_rate":       "LOINC/8867-4",    # beats/min
    "respiratory_rate": "LOINC/9279-1",    # breaths/min
    "body_temperature": "LOINC/8310-5",    # Cel (or [degF])
    "spo2":             "LOINC/2708-6",    # %   (use LOINC/59408-5 for the pulse-ox-specific concept)
    "body_height":      "LOINC/8302-2",    # cm  (or [in_us])
    "body_weight":      "LOINC/29463-7",   # kg  (or [lb_av])
    "bmi":              "LOINC/39156-5",   # kg/m2
}


def write_patient(writer, patient_id, birth_date, gender_code, race_code, timeline):
    writer.writerow([patient_id, birth_date.isoformat(), DEMOGRAPHIC_CODES["birth"],
                     "", "", 1, "", "PATIENT"])
    writer.writerow([patient_id, birth_date.isoformat(), gender_code,
                     "", "", 1, "", "PATIENT"])     # gender is the CODE, value empty
    writer.writerow([patient_id, birth_date.isoformat(), race_code,
                     "", "", 1, "", "PATIENT"])     # race is the CODE, value empty
    for e in timeline:
        writer.writerow([
            patient_id, e["date"].isoformat(), e["code"],
            e.get("value", ""), e.get("dosage", ""),
            e.get("visit_ids", 1), e.get("lab_units", ""), e.get("clarity_source", ""),
        ])
    # --- clinical events ---
    for e in timeline:
        writer.writerow([
            patient_id,
            e["date"].isoformat(),
            e["code"],
            e.get("value", ""),
            e.get("dosage", ""),
            e.get("visit_ids", 1),
            e.get("lab_units", ""),
            e.get("clarity_source", ""),
        ])


def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    out_path = os.path.join(OUT_DIR, OUT_FILE)

    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            ["patient_id", "start", "code", "value", "dosage", "visit_ids", "lab_units", "clarity_source"]
        )

        # This is a patient with a BMI of 50, a previous stroke, and an incarcerated hernia with a previous failed repair
        d = datetime.date
        write_patient(writer, patient_id=900002, birth_date=d(1959, 3, 14),
                      gender_code="Gender/F", race_code="Race/3",   # Male, Black or African American
            timeline=[
                {"code": VITALS_CODES["systolic_bp"],   "date": d(2020, 7, 9), "value": 160.0, "lab_units": "mmHg", "clarity_source": "LAB_RESULT", "visit_ids": 9},
                {"code": VITALS_CODES["diastolic_bp"],   "date": d(2020, 7, 9), "value": 95.0,  "lab_units": "mmHg", "clarity_source": "LAB_RESULT", "visit_ids": 9},
                {"code": "SNOMED/230690007", "date": d(2010, 2, 1), "clarity_source": "DIAGNOSIS", "visit_ids": 1},
                #This next line is for HgA1C
                {"code": "LOINC/4548-4", "date": d(2020, 8, 9), "value": 10.0,   "lab_units": "%",    "clarity_source": "LAB_RESULT", "visit_ids": 9},
                {"code": VITALS_CODES["bmi"], "date": d(2019, 2, 1), "value": 50.0,  "lab_units": "kg/m2", "clarity_source": "LAB_RESULT", "visit_ids": 9},
                {"code": "CPT4/49652", "date": d(2015, 9, 22), "clarity_source": "PROCEDURES", "visit_ids": 2},
                {"code": "CPT4/49653", "date": d(2022, 6, 5), "clarity_source": "PROCEDURES", "visit_ids": 5},
            ],
        )

        # ---------------- Patient 2 ----------------
        write_patient(writer, patient_id=900001, birth_date=d(1970, 1, 7),
                      gender_code="Gender/F", race_code="Race/5",   # Female, White
            timeline=[
                {"code": DIAGNOSIS_CODES[1], "date": d(2019, 2, 1), "clarity_source": "DIAGNOSIS", "visit_ids": 10},
                {"code": VITALS_CODES["systolic_bp"], "date": d(2019, 2, 1), "value": 145.0, "lab_units": "mmHg", "clarity_source": "LAB_RESULT", "visit_ids": 10},
                {"code": VITALS_CODES["diastolic_bp"], "date": d(2019, 2, 1), "value": 90.0,  "lab_units": "mmHg", "clarity_source": "LAB_RESULT", "visit_ids": 10},
                {"code": DRUG_CODES[1],      "date": d(2019, 3, 10), "dosage": 500, "lab_units": "mg", "clarity_source": "MED_ORDER", "visit_ids": 13},
                {"code": "CPT4/49652", "date": d(2021, 9, 22), "clarity_source": "PROCEDURES", "visit_ids": 17},
            ],
        )

    # quick sanity check: every code has a "/"
    with open(out_path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            assert "/" in row["code"], f"BAD CODE (no '/'): {row['code']}"

    print(f"Wrote {out_path}")
    print("Next: run  etl_simple_femr  on the FOLDER containing this file (see README).")


if __name__ == "__main__":
    main()

Overwriting make_patients.py


In [ ]:
#Do this next
%%bash
python make_patients.py
# -> simple_femr_input/simulated_patients.csv

Wrote simple_femr_input/simulated_patients.csv
Next: run  etl_simple_femr  on the FOLDER containing this file (see README).


In [ ]:
#You have to point these commands to the femr environment (femr_venv) you've created using python 3.10 and femr 0.1.16
#etl_simple_femr here takes a folder of CSVs (in this case, the folder is simple_femr_input) and makes "my_extract"
!/content/femr_venv/bin/etl_simple_femr simple_femr_input my_extract /tmp/femr_simple_tmp --num_threads 1

2026-06-25 08:52:42,302 [MainThread  ] [INFO ]  Extracting from OMOP with arguments Namespace(simple_source='simple_femr_input', target_location='/content/my_extract', temp_location='/tmp/femr_simple_tmp', num_threads=1, athena_download=None)
2026-06-25 08:52:42,302 [MainThread  ] [INFO ]  Already converted to events, skipping
2026-06-25 08:52:42,302 [MainThread  ] [INFO ]  Already converted to patients, skipping
2026-06-25 08:52:42,302 [MainThread  ] [INFO ]  Already converted to extract, skipping


In [ ]:
#This takes my_extract and makes it into a femr.datasets.PatientDatabase, something the model can use
#I think I have to do this before computing representations; this makes our prediction times?
#The rule of thumb going forward is just that anything touching femr goes through /content/femr_venv/bin/
%%writefile make_pred_times.py
import csv, femr.datasets
db = femr.datasets.PatientDatabase("my_extract")
with open("prediction_times.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["patient_id", "prediction_time"])
    for pid in db:
        last = max(e.start for e in db[pid].events)
        w.writerow([pid, last.isoformat()])
print("wrote prediction_times.csv")



Overwriting make_pred_times.py


In [ ]:
!/content/femr_venv/bin/python make_pred_times.py

wrote prediction_times.csv


In [ ]:
#You will be using this to compute the representations; again, you're pointing it to your femr environment
#You have to use this PATH= thing because femr_compute_representations shells out to another script, and it can't
#find venv's bin/ on PATH for that child process otherwise.
!PATH="/content/femr_venv/bin:$PATH" /content/femr_venv/bin/femr_compute_representations \
  --data_path my_extract \
  --model_path /root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc \
  --prediction_times_path prediction_times.csv \
  --batch_size 1024 \
  motor_reprs.pkl

2026-06-25 08:57:02,819 [MainThread  ] [INFO ]  Preparing batches with Namespace(directory='/tmp/tmplk8chf8a/task_batches', data_path='my_extract', dictionary_path='/root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc/dictionary', task='labeled_patients', transformer_vocab_size=65536, clmbr_survival_dictionary_path=None, labeled_patients_path='prediction_times.csv', is_hierarchical=True, seed=97, val_start=70, test_start=85, batch_size=1024, note_embedding_data=None, limit_to_patients_file=None, limit_before_date=None, num_clmbr_tasks=8192)
2026-06-25 08:57:09,132 [MainThread  ] [INFO ]  Wrote config ...
2026-06-25 08:57:09,132 [MainThread  ] [INFO ]  Starting to load
When mapping codes, dropped 14174 out of 40996
When mapping codes, dropped 14174 out of 40996
2026-06-25 08:57:32,894 [MainThread  ] [INFO ]  Loaded
When mapping codes, dropped 14174 out of 40996
2026-06-25 08:57:45,152 [MainThread  ] [INFO ]  Number of trai

In [ ]:
!/content/femr_venv/bin/python inspect_motor_model.py /root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc

FILE TREE: /root/.cache/huggingface/hub/models--StanfordShahLab--motor-t-base/snapshots/71f506ce2f5de7e9291e990d02cf0b0ef56139dc
71f506ce2f5de7e9291e990d02cf0b0ef56139dc/
  .gitattributes   (0.00 MB)
  README.md   (0.01 MB)
  dictionary   (217.65 MB)
  model/
    best   (571.28 MB)
    best_info   (0.00 MB)
    best_test_loss   (0.00 MB)
    config.msgpack   (0.00 MB)

MODEL CONFIG  (model/config.msgpack)
task.type        : survival_clmbr
task.dim         : 512
task.num_codes   : 8192
task.num_time_bins: 8
transformer.vocab_size    : 65536
transformer.hidden_size   : 768
transformer.is_hierarchical: True
No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)

PARAM MODULES  (model/best)
EHRTransformer/~/SurvivalCLMBRTask
    code_weight: (8192, 511)
    code_weight_bias: (8192, 1)
EHRTransformer/~/SurvivalCLMBRTask/~/linear
    b: (4088,)
    w: (768, 4088)
EHRTransformer/~/TransformerFeaturizer/~/Transformer/~/embed
    embeddings: (65536, 768)
EH

In [ ]:
%%writefile inspect_motor_model.py
"""
inspect_motor_model.py
----------------------
Figure out whether motor-t-base's native time-to-event head can be used to read
out outcome hazards -- and whether the model ships the "survival dictionary"
(the ordered list of codes the model predicts) needed to LABEL those hazards.

Run with the femr venv Python:
    /content/femr_venv/bin/python inspect_motor_model.py /path/to/motor-t-base
"""

import os
import sys
import pickle

import msgpack


def show_tree(root):
    print("=" * 70)
    print("FILE TREE:", root)
    print("=" * 70)
    for dirpath, _dirs, files in os.walk(root):
        depth = dirpath[len(root):].count(os.sep)
        print("  " * depth + os.path.basename(dirpath.rstrip(os.sep)) + "/")
        for fn in sorted(files):
            full = os.path.join(dirpath, fn)
            try:
                sz = os.path.getsize(full)
            except OSError:
                sz = -1
            print("  " * (depth + 1) + f"{fn}   ({sz/1e6:.2f} MB)")


def load_msgpack(path):
    with open(path, "rb") as f:
        return msgpack.load(f, use_list=False, raw=False)


def show_config(model_path):
    cfg_path = os.path.join(model_path, "model", "config.msgpack")
    if not os.path.exists(cfg_path):
        print(f"\n!! no config.msgpack at {cfg_path}")
        return None
    cfg = load_msgpack(cfg_path)
    print("\n" + "=" * 70)
    print("MODEL CONFIG  (model/config.msgpack)")
    print("=" * 70)
    task = cfg.get("task", {})
    trans = cfg.get("transformer", {})
    print("task.type        :", task.get("type"))
    print("task.dim         :", task.get("dim"))
    print("task.num_codes   :", task.get("num_codes"))
    print("task.num_time_bins:", task.get("num_time_bins"))
    tb = task.get("time_bins")
    if tb is not None:
        print("task.time_bins   :", list(tb))
    print("transformer.vocab_size    :", trans.get("vocab_size"))
    print("transformer.hidden_size   :", trans.get("hidden_size"))
    print("transformer.is_hierarchical:", trans.get("is_hierarchical"))
    extra = {k: ("<{} items>".format(len(v)) if hasattr(v, "__len__") and not isinstance(v, str) else v)
             for k, v in task.items()
             if k not in ("type", "dim", "num_codes", "num_time_bins", "time_bins")}
    if extra:
        print("other task.* keys:", extra)
    return cfg


def show_params(model_path):
    best_path = os.path.join(model_path, "model", "best")
    if not os.path.exists(best_path):
        print(f"\n!! no params at {best_path}")
        return
    with open(best_path, "rb") as f:
        params = pickle.load(f)
    print("\n" + "=" * 70)
    print("PARAM MODULES  (model/best)")
    print("=" * 70)
    survival_found = False
    for module in sorted(params.keys()):
        if "Survival" in module:
            survival_found = True
        weights = params[module]
        shapes = {name: getattr(v, "shape", None) for name, v in weights.items()}
        print(f"{module}")
        for name, shp in shapes.items():
            print(f"    {name}: {shp}")
    print("-" * 70)
    print("SurvivalCLMBRTask head present:", survival_found,
          "  <- needed to compute hazards")


def hunt_survival_dictionary(model_path):
    print("\n" + "=" * 70)
    print("SEARCHING FOR A SURVIVAL DICTIONARY / CODE LIST")
    print("=" * 70)
    candidates = []
    for dirpath, _dirs, files in os.walk(model_path):
        for fn in files:
            low = fn.lower()
            if "surv" in low or "dict" in low or fn.endswith(".msgpack"):
                candidates.append(os.path.join(dirpath, fn))

    found_codes = False
    for path in candidates:
        print(f"\n- {path}")
        try:
            obj = load_msgpack(path)
        except Exception as e:
            print(f"    (not msgpack-loadable: {e})")
            continue
        if isinstance(obj, dict):
            keys = list(obj.keys())
            print(f"    top-level keys: {keys[:20]}")
            for key in ("codes", "survival_dict", "lambdas", "time_bins"):
                if key in obj:
                    v = obj[key]
                    n = len(v) if hasattr(v, "__len__") else "?"
                    print(f"    -> contains {key!r} (len {n})")
                    if key == "codes":
                        found_codes = True
                        print(f"       first codes: {list(v)[:10]}")
            sd = obj.get("survival_dict")
            if isinstance(obj.get("task"), dict):
                sd = sd or obj["task"].get("survival_dict")
            if isinstance(sd, dict) and "codes" in sd:
                found_codes = True
                print(f"    -> nested survival_dict.codes (len {len(sd['codes'])})")
                print(f"       first codes: {list(sd['codes'])[:10]}")

    print("-" * 70)
    if found_codes:
        print("RESULT: found a code list -> hazards CAN be mapped to clinical codes.")
    else:
        print("RESULT: no survival code list found in the model dir.")
        print("Without it, hazards are computable but unlabeled.")


def main():
    if len(sys.argv) < 2:
        print(__doc__); sys.exit(1)
    model_path = sys.argv[1]
    show_tree(model_path)
    show_config(model_path)
    show_params(model_path)
    hunt_survival_dictionary(model_path)


if __name__ == "__main__":
    main()

Overwriting inspect_motor_model.py


In [ ]:
#This is how you could inspect output
import pickle
r = pickle.load(open("motor_reprs.pkl", "rb"))
print(r.keys())
print("representations", r["representations"].shape)   # (2, repr_dim)
print("patient_ids", r["patient_ids"])                 # 900001, 900002
print("prediction_times", r["prediction_times"])

dict_keys(['representations', 'patient_ids', 'prediction_times'])
representations (2, 769)
patient_ids [900001 900002]
prediction_times [datetime.datetime(2021, 9, 22, 0, 0) datetime.datetime(2022, 6, 5, 0, 0)]


In [ ]:
%%writefile check_representations.py
"""
check_representations.py
------------------------
Sanity-check a motor_reprs.pkl from femr_compute_representations to decide whether
the embeddings are real or degenerate.

Runs fine in the normal Colab kernel (only needs numpy + pickle -- no femr):
    python check_representations.py motor_reprs.pkl

"Degenerate" warning signs this checks for:
  * NaNs / infinities
  * all-zero or near-zero rows (no signal)
  * dimensions that are constant across all rows (carry no information)
  * two patients whose embeddings are essentially identical (cosine ~ 1.0)
    even though their timelines differ
"""

import sys
import pickle
import numpy as np


def main():
    path = sys.argv[1] if len(sys.argv) > 1 else "motor_reprs.pkl"
    with open(path, "rb") as f:
        r = pickle.load(f)

    print("keys:", list(r.keys()))
    reps = np.asarray(r["representations"], dtype=float)
    pids = np.asarray(r["patient_ids"])
    times = r.get("prediction_times")
    n, dim = reps.shape
    print(f"representations shape: {reps.shape}  ({n} prediction times, dim={dim})")
    print("patient_ids:", pids.tolist())
    if times is not None:
        print("prediction_times:", list(times)[:n])
    print("-" * 70)

    issues = []

    # 1. NaN / inf
    n_nan = int(np.isnan(reps).sum())
    n_inf = int(np.isinf(reps).sum())
    print(f"NaNs: {n_nan}   infs: {n_inf}")
    if n_nan or n_inf:
        issues.append("contains NaN/inf")

    # 2. per-row norms (exclude a possible constant bias column at the end)
    norms = np.linalg.norm(reps, axis=1)
    print("per-row L2 norm:", np.round(norms, 4).tolist())
    if np.any(norms < 1e-6):
        issues.append("one or more rows are all-zero")

    # 3. constant dimensions (zero variance across rows)
    col_var = reps.var(axis=0)
    n_const = int((col_var < 1e-12).sum())
    print(f"constant dimensions (zero variance across rows): {n_const} / {dim}")
    # note: 1 constant dim is normal -- MOTOR appends a constant bias/intercept term
    if n_const > max(1, dim // 2):
        issues.append(f"{n_const}/{dim} dims are constant -- little information")

    # 4. distinctness between rows (cosine similarity)
    if n >= 2:
        # cosine similarity matrix
        unit = reps / np.clip(norms[:, None], 1e-12, None)
        cos = unit @ unit.T
        print("-" * 70)
        print("pairwise cosine similarity:")
        np.set_printoptions(precision=4, suppress=True)
        print(cos)
        # flag near-identical distinct rows
        iu = np.triu_indices(n, k=1)
        max_offdiag = cos[iu].max() if iu[0].size else 0.0
        print(f"max off-diagonal cosine (closest two rows): {max_offdiag:.4f}")
        if max_offdiag > 0.999:
            issues.append("two rows are nearly identical (cosine > 0.999) -- "
                          "patients may not be differentiated")
    print("-" * 70)

    # verdict
    if issues:
        print("POTENTIALLY DEGENERATE:")
        for i in issues:
            print("  -", i)
    else:
        print("LOOKS HEALTHY: non-zero norms, varied dimensions, and rows are "
              "distinguishable from each other.")
        print("(With only a couple of patients this is a smoke test, not proof of "
              "clinical validity -- but it rules out the obvious failure modes.)")


if __name__ == "__main__":
    main()

Writing check_representations.py


In [ ]:
#What to look for in its output:
#per-row L2 norm — should be clearly non-zero (a row of ~0 means no signal).
#constant dimensions — 1 is normal (MOTOR appends a constant bias term); a large fraction being constant is a red flag.
#pairwise cosine similarity — your two patients have different timelines, so the off-diagonal should be well below 1.0.
#If it's ~0.999, the model isn't distinguishing them.
#Final line gives a plain verdict.

!python check_representations.py motor_reprs.pkl

keys: ['representations', 'patient_ids', 'prediction_times']
representations shape: (2, 769)  (2 prediction times, dim=769)
patient_ids: [900001, 900002]
prediction_times: [datetime.datetime(2021, 9, 22, 0, 0), datetime.datetime(2022, 6, 5, 0, 0)]
----------------------------------------------------------------------
NaNs: 0   infs: 0
per-row L2 norm: [28.6607, 28.8623]
constant dimensions (zero variance across rows): 1 / 769
----------------------------------------------------------------------
pairwise cosine similarity:
[[1.     0.6042]
 [0.6042 1.    ]]
max off-diagonal cosine (closest two rows): 0.6042
----------------------------------------------------------------------
LOOKS HEALTHY: non-zero norms, varied dimensions, and rows are distinguishable from each other.
(With only a couple of patients this is a smoke test, not proof of clinical validity -- but it rules out the obvious failure modes.)


In [ ]:
%%bash
cat > /content/femr_venv/lib/python3.10/site-packages/sitecustomize.py << 'EOF'
import jax.numpy as jnp
jnp.DeviceArray = jnp.ndarray
EOF
cat /content/femr_venv/lib/python3.10/site-packages/sitecustomize.py

import jax.numpy as jnp
jnp.DeviceArray = jnp.ndarray


In [ ]:
%%bash
sudo bash -c 'cat >> /usr/lib/python3.10/sitecustomize.py << "EOF"

try:
    import jax.numpy as jnp
    jnp.DeviceArray = jnp.ndarray
except ImportError:
    pass
EOF'
cat /usr/lib/python3.10/sitecustomize.py

# install the apport exception handler if available
try:
    import apport_python_hook
except ImportError:
    pass
else:
    apport_python_hook.install()

try:
    import jax.numpy as jnp
    jnp.DeviceArray = jnp.ndarray
except ImportError:
    pass

try:
    import types
    import jax
    if not hasattr(jax, "xla"):
        _fake_xla = types.ModuleType("jax.xla")
        _fake_xe = types.ModuleType("jax.xla.xe")
        class _DummyJaxClass:
            pass
        _fake_xe.CompiledFunction = _DummyJaxClass
        _fake_xe.PmapFunction = _DummyJaxClass
        _fake_xla.xe = _fake_xe
        jax.xla = _fake_xla
except ImportError:
    pass

try:
    import jax.numpy as jnp
    jnp.DeviceArray = jnp.ndarray
except ImportError:
    pass

try:
    import types
    import jax
    if not hasattr(jax, "xla"):
        _fake_xla = types.ModuleType("jax.xla")
        _fake_xe = types.ModuleType("jax.xla.xe")
        class _DummyJaxClass:
            pass
        _fake_xe.CompiledFunction 

In [ ]:
!/content/femr_venv/bin/python -c "import jax.numpy as jnp; print(jnp.DeviceArray)"

<class 'jax.Array'>


In [ ]:
%%bash
sudo bash -c 'cat >> /usr/lib/python3.10/sitecustomize.py << "EOF"

try:
    import types
    import jax
    if not hasattr(jax, "xla"):
        _fake_xla = types.ModuleType("jax.xla")
        _fake_xe = types.ModuleType("jax.xla.xe")
        class _DummyJaxClass:
            pass
        _fake_xe.CompiledFunction = _DummyJaxClass
        _fake_xe.PmapFunction = _DummyJaxClass
        _fake_xla.xe = _fake_xe
        jax.xla = _fake_xla
except ImportError:
    pass
EOF'

In [ ]:
%%bash
export PATH="/content/femr_venv/bin:$PATH"
femr_compute_representations --data_path synthetic_data/extract_lite --model_path synthetic_data/motor --prediction_times_path prediction_times.csv --batch_size 1024 motor_reprs.pkl

When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out of 2048
0 out of 2048
When mapping codes, dropped 0 out of 2048
When mapping codes, dropped 0 out 

2026-06-25 03:48:00,341 [MainThread  ] [INFO ]  Preparing batches with Namespace(directory='/tmp/tmpip7votb3/task_batches', data_path='synthetic_data/extract_lite', dictionary_path='synthetic_data/motor/dictionary', task='labeled_patients', transformer_vocab_size=2048, clmbr_survival_dictionary_path=None, labeled_patients_path='prediction_times.csv', is_hierarchical=False, seed=97, val_start=70, test_start=85, batch_size=1024, note_embedding_data=None, limit_to_patients_file=None, limit_before_date=None, num_clmbr_tasks=8192)
2026-06-25 03:48:00,368 [MainThread  ] [INFO ]  Wrote config ...
2026-06-25 03:48:00,368 [MainThread  ] [INFO ]  Starting to load
2026-06-25 03:48:00,660 [MainThread  ] [INFO ]  Loaded
2026-06-25 03:48:00,721 [MainThread  ] [INFO ]  Number of train patients 5


In [ ]:
!ls -la motor_reprs.pkl

-rw-r--r-- 1 root root 212448 Jun 25 03:48 motor_reprs.pkl


In [ ]:
%%bash
/content/femr_venv/bin/python -c "
import pickle
with open('motor_reprs.pkl', 'rb') as f:
    motor_reprs = pickle.load(f)

print(motor_reprs.keys())
print('-' * 80)
for k, v in motor_reprs.items():
    print(k, v.shape)
print('-' * 80)
for k, v in motor_reprs.items():
    if len(v.shape) == 2:
        print(k, v[:10, :])
    else:
        print(k, v[:10])
"

dict_keys(['representations', 'patient_ids', 'prediction_times'])
--------------------------------------------------------------------------------
representations (392, 257)
patient_ids (392,)
prediction_times (392,)
--------------------------------------------------------------------------------
representations [[ 0.1548  0.1927  1.495  ...  1.805  -0.927   1.    ]
 [ 0.64   -1.572   0.3352 ... -0.4333 -1.084   1.    ]
 [-0.8384  0.31    1.224  ... -0.2367  1.393   1.    ]
 ...
 [-0.8564  1.88    0.6196 ...  0.317   0.2336  1.    ]
 [ 1.054   0.2644 -0.1779 ... -1.586  -1.703   1.    ]
 [ 1.054   0.2644 -0.1779 ... -1.586  -1.703   1.    ]]
patient_ids [  3   3 100 100 101 101 102 102 103 103]
prediction_times [datetime.datetime(2020, 8, 9, 0, 0) datetime.datetime(2022, 6, 5, 0, 0)
 datetime.datetime(1991, 3, 3, 0, 0) datetime.datetime(1992, 5, 26, 0, 0)
 datetime.datetime(1992, 11, 2, 0, 0) datetime.datetime(1993, 11, 9, 0, 0)
 datetime.datetime(1993, 3, 10, 0, 0) datetime.datetime(1

In [ ]:
import pickle
import random
import csv
import numpy as np
import scipy

with open('motor_reprs.pkl', 'rb') as f:
    motor_reprs = pickle.load(f)

random.seed(4533421)
np.random.seed(3421342)
representation_size = motor_reprs['representations'].shape[1]
weight = np.random.normal(size=(representation_size,))
logits = np.dot(motor_reprs['representations'], weight)
probabilities = scipy.special.expit(logits)
sampled_probabilties = np.random.rand(len(logits))
labels = sampled_probabilties < probabilities
with open('labels.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerow(['patient_id', 'prediction_time', 'value'])
    for patient_id, prediction_time, label in zip(motor_reprs['patient_ids'], motor_reprs['prediction_times'], labels):
        writer.writerow([patient_id, prediction_time, label])

In [ ]:
import datetime
import csv
import pickle
import numpy as np

# Reload motor_reprs since it doesn't carry over between cells/sessions reliably
with open('motor_reprs.pkl', 'rb') as f:
    motor_reprs = pickle.load(f)

# Quick sanity checks (Python equivalents of !wc -l and !head)
with open('labels.csv') as f:
    lines = f.readlines()
print(len(lines))
print(''.join(lines[:10]))

# Load the labels into a Python dictionary
label_map = {}
with open('labels.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        label_map[int(row['patient_id']), datetime.datetime.fromisoformat(row['prediction_time'])] = row['value'] == 'True'

# Align the labels with our representations
labels = []
for patient_id, prediction_time in zip(motor_reprs['patient_ids'], motor_reprs['prediction_times']):
    labels.append(label_map[patient_id, prediction_time])
print(len(labels))
labels = np.array(labels)

393
patient_id,prediction_time,value
3,2020-08-09 00:00:00,True
3,2022-06-05 00:00:00,False
100,1991-03-03 00:00:00,True
100,1992-05-26 00:00:00,False
101,1992-11-02 00:00:00,True
101,1993-11-09 00:00:00,True
102,1993-03-10 00:00:00,True
102,1994-02-05 00:00:00,False
103,1993-08-06 00:00:00,True

392
